# Non-Equilibrium Transport Sampler
In this notebook, we train non-equilibrium transport samplers (NETS) [1] using overdamped and underdamped/inertial Langevin proposals on linear and interpolating annealing paths between a Gaussian an a 40-mode Gaussian mixture target [2].

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import torch
from omegaconf import OmegaConf
from matplotlib import pyplot as plt

from src.models.nets import NETSModule
from src.utils.train import train_module
from src.utils.plotting import plot_proposal, timeseries_plot_density, timeseries_plot_contours

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {device}')


In [2]:
# Visualize untrained vector field and untrained proposal
def plotter_helper(pinn_module, proposal_names, proposal_labels):
    scale = 50.0
    timesteps = torch.linspace(0,1,360).to(device)
    num_samples = 500

    num_proposals = len(proposal_names)
    _, axes = plt.subplots(num_proposals, 4, figsize=(16, 8), sharex=True, sharey=True)
    axes = axes.reshape((num_proposals, 4))

    proposals = {
        label: pinn_module.build_proposal(name).to(device) for name, label in zip(proposal_names, proposal_labels)
    }

    for i, (label, proposal) in enumerate(proposals.items()):
        proposal.record_every = 120
        axes[i, 0].set_ylabel(label, fontsize=15)

        # Plot
        _, plotted_timesteps = plot_proposal(
            proposal=proposal,
            num_samples=num_samples,
            ts=timesteps,
            axes=axes[i],
            scale=scale,
            use_alpha=False,
        )
        timeseries_plot_density(pinn_module.density_path.log_density, plotted_timesteps, scale=scale, axes=axes[i], alpha=0.5, vmin=-40, zorder=0, device=device)
        timeseries_plot_contours(pinn_module.density_path.log_density, plotted_timesteps, scale=scale, axes=axes[i], alpha=0.5, legend=False, levels=40, device=device)

### Overdamped NETS on Interpolant Path


In [3]:
# Initialize NETS module from config
overdamped_interpolant_cfg = OmegaConf.create(
    {
        # Run Details
        "run_name": "overdamped_interpolant_notebook_test",
        "run_group": "notebook_test",
        "wandb": False,
        "checkpoint": None,

        # Objective Details
        "x_dim": 2,
        "density_path": "interpolant",
        "target": "fab_gmm",
        "cov_scale": 1.0,
        "source_std": 2.0,

        # Training details
        "num_devices": 1,
        "max_epochs": 25,
        "lr": 0.002,
        "lr_burn_in_epochs": 0,
        "step_size": 10,
        "gamma": 0.97,
        "checkpoint_burn_in_epochs": 0,

        # Batching details
        "use_persistent_sample_buffer": True,
        "persistent_sample_buffer_trajectories": 500,
        "train_batch_size": 10000,
        "train_trajectories": 400,
        "train_steps_per_epoch": 50,
        "val_batch_size": 50000,
        "val_trajectories": 1000,
        "val_steps_per_epoch": 1,
        "val_freq": 5,

        # PINN-specific training details
        "annealing_scheduler": "constant",
        "start_T": 1.0,
        "avg_dt": 0.002,
        "record_every": 5,
        "divergence_mode": "autograd",

        # Control-specific details
        "control": "mlp",
        "control_hiddens": [256, 256, 256],

        # Free-energy specific details
        "free_energy": "mlp",
        "free_energy_hiddens": [256, 256, 256],

        # Time-encoding details
        "use_fourier": True,
        "x_fourier_dim": 100,
        "x_fourier_sigma": 0.1,
        "b_fourier_dim": 20,
        "b_fourier_sigma": 1,
        "t_fourier_dim": 20,
        "t_fourier_sigma": 5,
        
        # Proposal details
        "proposal": "overdamped_langevin",
        "damping": 50,
        "use_jarzynski": True,

        # Misc
        "verbose": False,
        "memory_profile": False,
    }
)

overdamped_interpolant_module = NETSModule(cfg=overdamped_interpolant_cfg).move_to_device(device)

In [ ]:
plotter_helper(
    pinn_module=overdamped_interpolant_module,
    proposal_names=["ode", "overdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

In [ ]:
# Train using replay buffer
train_module(
    module=overdamped_interpolant_module,
    run_name=overdamped_interpolant_cfg.run_name,
    run_group=overdamped_interpolant_cfg.run_group,
    max_epochs=overdamped_interpolant_cfg.max_epochs,
    val_freq=5,
    wandb=False,
)

In [ ]:
plotter_helper(
    pinn_module=overdamped_interpolant_module,
    proposal_names=["ode", "overdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

### Overdamped NETS on Linear Path

In [15]:
# Initialize NETS module from config
overdamped_linear_cfg = OmegaConf.create(
    {
    # Run Details
    "run_name": "overdamped_linear_notebook_test",
    "run_group": "notebook_test",
    "wandb": False,
    "checkpoint": None,

    # Objective Details
    "x_dim": 2,
    "density_path": "learnable_linear",
    "learnable_hiddens": [256, 256, 256],
    "target": "fab_gmm",
    "cov_scale": 1.0,
    "source_std": 5.0,

    # Training details
    "num_devices": 1,
    "max_epochs": 250,
    "lr": 0.002,
    "lr_burn_in_epochs": 0,
    "step_size": 5,
    "gamma": 0.97,
    "checkpoint_burn_in_epochs": 0,

    # Batching details
    "use_persistent_sample_buffer": True,
    "persistent_sample_buffer_trajectories": 2000,
    "train_trajectories": 250,
    "train_steps_per_epoch": 50,
    "val_trajectories": 1000,
    "val_steps_per_epoch": 2,
    "val_freq": 5,

    # PINN-specific training details
    "annealing_scheduler": "manual",
    "T_schedule": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "epochs_per_T": [10, 10, 10, 10, 20, 20, 20, 30, 30, 30],
    "avg_dt": 0.002,
    "record_every": 20,
    "divergence_mode": "autograd",

    # Control-specific details
    "control": "mlp",
    "control_hiddens": [256, 256, 256],

    # Free-energy specific details
    "free_energy": "mlp",
    "free_energy_hiddens": [256, 256, 256],

    # Time-encoding details
    "use_fourier": True,
    "x_fourier_dim": 100,
    "x_fourier_sigma": 0.1,
    "t_fourier_dim": 20,
    "t_fourier_sigma": 5,

    # Proposal details
    "proposal": "overdamped_langevin",
    "damping": 50,
    "use_jarzynski": True,

    # Misc
    "verbose": False,
    "memory_profile": False,
    }
)

overdamped_linear_module = NETSModule(cfg=overdamped_linear_cfg).move_to_device(device)

In [ ]:
# Visualize untrained vector field and untrained proposal
plotter_helper(
    pinn_module=overdamped_linear_module,
    proposal_names=["ode", "overdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

In [ ]:
# Train using replay buffer
train_module(
    module=overdamped_interpolant_module,
    run_name=overdamped_linear_cfg.run_name,
    run_group=overdamped_linear_cfg.run_group,
    max_epochs=overdamped_linear_cfg.max_epochs,
    val_freq=5,
    wandb=False,
)

In [ ]:
# Visualize trained vector field and proposal
plotter_helper(
    pinn_module=overdamped_linear_module,
    proposal_names=["ode", "overdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

### Inertial NETS on Interpolant Path


In [30]:
# Initualize NETS module
inertial_interpolant_cfg = OmegaConf.create(
    {
        # Run Details
        "run_name": "inertial_interpolant_notebook_test",
        "run_group": "notebook_test",
        "wandb": False,
        "checkpoint": None,

       # Objective Details
        "x_dim": 2,
        "density_path": "interpolant",
        "target": "fab_gmm",
        "cov_scale": 1.0,
        "source_std": 2.0,

        # Training details
        "num_devices": 1,
        "max_epochs": 25,
        "lr": 0.002,
        "lr_burn_in_epochs": 0,
        "step_size": 5,
        "gamma": 0.97,
        "checkpoint_burn_in_epochs": 0,

        # Batching details
        "use_persistent_sample_buffer": True,
        "persistent_sample_buffer_trajectories": 500,
        "train_batch_size": 40000,
        "train_trajectories": 400,
        "train_steps_per_epoch": 50,
        "val_batch_size": 50000,
        "val_trajectories": 1000,
        "val_steps_per_epoch": 1,
        "val_freq": 5,

        # PINN-specific training details
        "annealing_scheduler": "constant",
        "start_T": 1.0,
        "avg_dt": 0.002,
        "record_every": 5,
        "divergence_mode": "autograd",

        # Control-specific details
        "control": "mlp",
        "control_hiddens": [256, 256, 256],

        # Free-energy specific details
        "free_energy": "mlp",
        "free_energy_hiddens": [256, 256, 256],

        # Encoding details
        "use_fourier": True,
        "x_fourier_dim": 100,
        "x_fourier_sigma": 0.1,
        "t_fourier_dim": 20,
        "t_fourier_sigma": 5,

        # Proposal details
        "proposal": "underdamped_langevin",
        "scale": 50.0,
        "damping": 2.0,
        "mass": 1.0,
        "hamiltonian_type": "standard",
        "use_jarzynski": True,

        # Misc
        "verbose": False,
        "memory_profile": False, 
    }
)

inertial_interpolant_module = NETSModule(cfg=inertial_interpolant_cfg).move_to_device(device)

In [ ]:
# Visualize untrained vector field and untrained proposal
plotter_helper(
    pinn_module=inertial_interpolant_module,
    proposal_names=["ode", "underdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

In [ ]:
# Train using replay buffer
train_module(
    module=inertial_interpolant_module,
    run_name=inertial_interpolant_cfg.run_name,
    run_group=inertial_interpolant_cfg.run_group,
    max_epochs=inertial_interpolant_cfg.max_epochs,
    val_freq=5,
    wandb=False,
)

In [ ]:
# Visualize trained transport and trained proposal
plotter_helper(
    pinn_module=inertial_interpolant_module,
    proposal_names=["ode", "underdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

### Inertial NETS on Linear Path

In [ ]:
# Initualize NETS module
inertial_linear_cfg = OmegaConf.create(
    {
        # Run Details
        "run_name": "inertial_linear_notebook_test",
        "run_group": "notebook_test",
        "wandb": False,
        "checkpoint": None,

        # Objective Details
        "x_dim": 2,
        "density_path": "learnable_linear",
        "learnable_hiddens": [256, 256, 256],
        "target": "fab_gmm",
        "cov_scale": 1.0,
        "source_std": 5.0,

        # Training details
        "num_devices": 1,
        "max_epochs": 250,
        "lr": 0.002,
        "lr_burn_in_epochs": 0,
        "step_size": 5,
        "gamma": 0.97,
        "checkpoint_burn_in_epochs": 0,

        # Batching details
        "use_persistent_sample_buffer": True,
        "persistent_sample_buffer_trajectories": 2000,
        "train_trajectories": 250,
        "train_steps_per_epoch": 50,
        "val_trajectories": 1000,
        "val_steps_per_epoch": 2,
        "val_freq": 5,

        # PINN-specific training details
        "annealing_scheduler": "manual",
        "T_schedule": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "epochs_per_T": [10, 10, 10, 10, 20, 20, 20, 30, 30, 30],
        "avg_dt": 0.002,
        "record_every": 20,
        "divergence_mode": "autograd",

        # Control-specific details
        "control": "mlp",
        "control_hiddens": [256, 256, 256],

        # Free-energy specific details
        "free_energy": "mlp",
        "free_energy_hiddens": [256, 256, 256],

        # Time-encoding details
        "use_fourier": True,
        "x_fourier_dim": 100,
        "x_fourier_sigma": 0.1,
        "t_fourier_dim": 20,
        "t_fourier_sigma": 5,

        # Proposal details
        "proposal": "underdamped_langevin",
        "scale": 50.0,
        "damping": 2.0,
        "mass": 1.0,
        "hamiltonian_type": "standard",
        "use_jarzynski": True,

        # Misc
        "verbose": False,
        "memory_profile": False, 
    }
)

inertial_linear_module = NETSModule(cfg=inertial_linear_cfg).move_to_device(device)

In [ ]:
# Visualize untrained transport and proposal
plotter_helper(
    pinn_module=inertial_linear_module,
    proposal_names=["ode", "underdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

In [ ]:
# Train using replay buffer
train_module(
    module=inertial_linear_module,
    run_name=inertial_linear_cfg.run_name,
    run_group=inertial_linear_cfg.run_group,
    max_epochs=inertial_linear_cfg.max_epochs,
    val_freq=5,
    wandb=False,
)

In [ ]:
# Visualize trained transport and trained proposal
plotter_helper(
    pinn_module=inertial_linear_module,
    proposal_names=["ode", "underdamped_langevin"],
    proposal_labels=["Transport", "Transport + Langevin"]
)

### References
1. M. S. Albergo and E. Vanden-Eijnden. Nets: A non-equilibrium transport sampler. arXiv preprint
arXiv:2410.02711, 2024.
2. Laurence Illing Midgley, Vincent Stimper, Gregor N. C. Simm, Bernhard Schölkopf, and José Miguel
Hernández-Lobato. Flow annealed importance sampling bootstrap. In The Eleventh International
Conference on Learning Representations, 2023. URL https://openreview.net/forum?id=XCTVFJwS9LJ.